# Fine-Tuning QLoRA (Agente Único Tool-Calling + RAG + PDFs) → Ollama `Q4_K_M`

Este notebook entrena **1 solo adaptador QLoRA quirúrgico** sobre **`Qwen/Qwen3-4B-Instruct-2507`** (el modelo exacto detrás de `qwen3:4b-instruct` en Ollama) para correr finalmente en tu Mac de **8 GB de RAM**.

### ¿Qué lo diferencia de los LoRAs anteriores de D4?
1. **Base exacta `Qwen/Qwen3-4B-Instruct-2507`** con plantilla nativa `<tools>` / `<tool_call>` / `<tool_response>`.
2. **Cero fuga de datos (`antifuga.py`):** bloquea las 61 unidades evaluadas en `congelado_50`, `dificil_25` y `nuevas_20` (más vecinas cruzadas) y verifica similitud contra las 95 preguntas.
3. **Entrenamiento multi-turno realista (`labels = -100`):** solo calcula pérdida sobre los `tool_calls` y la respuesta final posterior al último mensaje `user` (nunca memoriza `system`, `historial` ni `<tool_response>`).
4. **Conversión directa a `GGUF Q4_K_M` + `Modelfile` Ollama** preservando `Capabilities: tools`.

In [ ]:
#@title 1. GPU, repositorio (`exp/finetune-agente`) y dependencias
import os, subprocess, sys
from pathlib import Path

!nvidia-smi

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
REPO_BRANCH = "exp/finetune-agente"
REPO_DIR = Path("/content/Proyecto_Git")

if not REPO_DIR.exists():
    res = subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Asegúrate de haber hecho push de la rama {REPO_BRANCH} a GitHub antes de ejecutar en Colab.")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)

os.chdir(REPO_DIR)
!pip install -q "transformers>=4.51.0" "peft>=0.15.0" "bitsandbytes>=0.45.0" "accelerate>=1.0.0" gguf sentencepiece protobuf matplotlib pymupdf pdfplumber


In [ ]:
#@title 2. Regenerar dataset sin fugas, correr tests unitarios y validar tokenización multi-turno
!python3 -m asistente_local.finetune.construir_dataset
!python3 -m unittest discover -s "asistente_local/tests" -v
!python3 -m asistente_local.finetune.entrenar_agente_sft --validate-only


In [ ]:
#@title 3. Entrenar QLoRA (`lr=5e-5`, 3 épocas, dropout 0.08) y fusionar pesos en FP16/BF16
from IPython.display import Image, display
from pathlib import Path

OUT_DIR = Path("/content/ft_salida")
MERGED_DIR = Path("/content/ft_merged_hf")

!python3 -m asistente_local.finetune.entrenar_agente_sft \
    --output-dir /content/ft_salida \
    --merge-dir /content/ft_merged_hf \
    --epochs 3.0 \
    --lr 5e-5 \
    --grad-accum 4

if (OUT_DIR / "loss_curve.png").is_file():
    display(Image(filename=str(OUT_DIR / "loss_curve.png")))
print(Path(OUT_DIR / "training_metrics.json").read_text())


In [ ]:
#@title 4. Compilar solo `llama-quantize` (CPU, ~30s) y exportar GGUF Q4_K_M + Modelfile Ollama
from pathlib import Path

LLAMA_CPP = Path("/content/llama.cpp")
if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -B /content/llama.cpp/build /content/llama.cpp -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j4

!python3 -m asistente_local.finetune.exportar_ollama \
    --merged-dir /content/ft_merged_hf \
    --salida-dir /content/ft_ollama_pkg \
    --llama-cpp-dir /content/llama.cpp \
    --nombre-base udec-agente-qwen3-4b-ft \
    --cuantizacion Q4_K_M

!ls -lh /content/ft_ollama_pkg


In [ ]:
#@title 5. Guardar en Google Drive (y opcionalmente en Hugging Face) para descargar al Mac
import shutil
from pathlib import Path

USAR_GOOGLE_DRIVE = True
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/UdeC_GenIA_Finetune_Agente")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree("/content/ft_ollama_pkg", dest / "ollama_pkg", dirs_exist_ok=True)
    shutil.copytree("/content/ft_salida/adapter", dest / "adapter", dirs_exist_ok=True)
    shutil.copy("/content/ft_salida/training_metrics.json", dest / "training_metrics.json")
    shutil.copy("/content/ft_salida/loss_curve.png", dest / "loss_curve.png")
    print("Guardado en Google Drive:", dest)

# En tu Mac, descarga `ollama_pkg/` y ejecuta:
#   cd ollama_pkg && ollama create udec-agente:4b-ft -f Modelfile
# Luego evalúalo localmente contra el base en el mismo código:
#   python3 -m asistente_local.benchmark --dataset nuevas_20 --modelo udec-agente:4b-ft
#   python3 -m asistente_local.benchmark --dataset dificil_25 --modelo udec-agente:4b-ft
